# Step 17: Intermediate: Context Engineering and Memory Patterns

        _Learner Notebook_  
        Level: **Intermediate**  
        Estimated time: **90 minutes**

        ## Learning Objectives
        - [ ] Separate active session state from durable memory and injected context.
- [ ] Build prompt-ready context blocks from stored history.
- [ ] Introduce summarization and pruning strategies for long conversations.
- [ ] Reason about what should and should not be remembered.

        ## Prerequisites
        - Core Steps 3 and 8 completed.
- Comfort with basic session management and SQLite examples.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Reflection & Next Experiments
8. Summary & Key Takeaways
9. Additional Resources


## Setup & Imports

Supplemental notebooks assume the core helpers are available and that you have already worked
through the matching core lessons. The import cell intentionally keeps the same bootstrap shape
as the core course so you can focus on the deeper pattern rather than environment setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Context engineering is often more important than adding another model call. This notebook focuses on what context to keep, how to compress it, and how to inject it without polluting the conversation.

This notebook is intentionally deeper than the core path. Expect more design discussion,
more implementation sections, and more open-ended exercises.


## Part 2: Core Implementation

### Create separate short-term and long-term stores


In [ ]:
from agent_framework import AgentSession

session = AgentSession(session_id="context-lab")
memory = SQLiteConversationMemory(resolve_notebook_root() / "data" / "databases" / "context_lab.sqlite3")

context_agent = create_agent(
    name="ContextAgent",
    instructions="You are a context-aware assistant. Prefer concise answers and use prior notes only when relevant.",
)


## Part 2: Core Implementation

### Capture durable memory candidates


In [ ]:
interactions = [
    ("sam", "My name is Sam and I lead platform tooling."),
    ("sam", "I prefer examples in Python, not pseudocode."),
    ("sam", "Please keep explanations short unless I ask for depth."),
]

for user_id, prompt in interactions:
    reply = await context_agent.run(prompt, session=session)
    memory.save(user_id, prompt, reply.text)

print_json([row.__dict__ for row in memory.history("sam", limit=5)], title="Persisted memory rows")


## Part 2: Core Implementation

### Build a reusable context injector


In [ ]:
def build_context_block(user_id: str, *, limit: int = 3) -> str:
    rows = list(reversed(memory.history(user_id, limit=limit)))
    if not rows:
        return ""
    bullets = "\n".join(f"- {row.user_message}" for row in rows)
    return f"Known user context:\n{bullets}"

context_block = build_context_block("sam")
print(context_block)


## Part 2: Core Implementation

### Summarize and prune older conversation details


In [ ]:
def summarize_history(user_id: str, *, limit: int = 5) -> dict:
    rows = list(reversed(memory.history(user_id, limit=limit)))
    summary = {
        "identity": [],
        "preferences": [],
        "current_focus": [],
    }
    for row in rows:
        text = row.user_message.lower()
        if "my name is" in text or "i lead" in text:
            summary["identity"].append(row.user_message)
        elif "prefer" in text or "keep explanations" in text:
            summary["preferences"].append(row.user_message)
        else:
            summary["current_focus"].append(row.user_message)
    return summary

print_json(summarize_history("sam"), title="Summarized memory view")


## Part 3: Hands-On Exercises

### Exercise 1
Write a helper that chooses whether a memory item belongs in identity, preference, or temporary-context buckets.

### Exercise 2
Create a prompt template that injects only preference memories, not every stored interaction.

Work through both guided exercises before comparing against the solutions.


In [ ]:
def classify_memory_item(text: str) -> str:
    # TODO: return one of: identity, preference, temporary
    return "temporary"


In [ ]:
# TODO: build a short prompt template that includes only preference notes


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
def classify_memory_item(text: str) -> str:
    lowered = text.lower()
    if "my name is" in lowered or "i lead" in lowered or "i work on" in lowered:
        return "identity"
    if "prefer" in lowered or "keep explanations" in lowered:
        return "preference"
    return "temporary"

print_json(
    {
        "name": classify_memory_item("My name is Sam."),
        "preference": classify_memory_item("I prefer Python examples."),
        "temporary": classify_memory_item("Can you summarize this issue?"),
    },
    title="Exercise 1 solution",
)


In [ ]:
summary = summarize_history("sam")
preference_prompt = (
    "Use the following stable preferences when you answer.\n"
    + "\n".join(f"- {item}" for item in summary["preferences"])
    + "\n\nUser request: Explain how workflow fan-out differs from branching."
)
print(preference_prompt)


## Part 5: Best Practices & Tips

        - Store only memory that is likely to matter again.
- Inject summarized context rather than raw transcripts whenever possible.
- Separate stable preferences from temporary task details.
- Treat context formatting as an explicit design problem, not a cleanup step.


## Reflection & Next Experiments

- Which part of `Intermediate: Context Engineering and Memory Patterns` felt like the biggest jump from the core course?
- What would you keep deterministic before turning this into a live production feature?
- Where would you add tests, traces, or operator controls before shipping this pattern?


## Summary & Key Takeaways

You deepened the memory story by shaping context deliberately instead of replaying whole conversations.


## Additional Resources

        - `helpers/memory.py`
- `core notebook: 08_memory_and_sessions.ipynb`
- `core notebook: 03_multi_turn_conversations.ipynb`
